# LLM Classification


### Classification with LLM

## Notebook overview

This notebook uses an LLM through OpenRouter to classify product reviews as `CG` fake/computer-generated or `OR` original/human-written. It loads the dataset, creates prompts, calls the model for a sample of reviews, and evaluates the predictions with a classification report and confusion matrix.


In [3]:
# Import libraries for file paths, data handling, and table display
from pathlib import Path
import pandas as pd
from IPython.display import display
from pqdm.threads import pqdm
import time
from openai import OpenAI
from sklearn.metrics import classification_report

resources_dir = Path("../resources")
dataset_path = resources_dir / "fake reviews dataset.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print(f"Dataset path: {dataset_path}")

Dataset path: ../resources/fake reviews dataset.csv


In [4]:
if not dataset_path.exists():
    print("Dataset not found locally. Running downloader.py...")
    !python downloader.py

    if not dataset_path.exists():
        raise FileNotFoundError(f"Downloader finished, but dataset is still missing: {dataset_path}")

    print(f"Dataset downloaded to: {dataset_path}")
else:
    print(f"Dataset already present: {dataset_path}")

Dataset already present: ../resources/fake reviews dataset.csv


In [5]:
# Load the project helper module and split the dataset into train and test parts
import base

train_df, test_df = base.load_and_split_data()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (32345, 4)
Test shape: (8087, 4)


In [6]:
display(train_df.head(13))
display(test_df.head())

,category,rating,label,text_
0,Kindle_Store_5,5.0,OR,I love paranormal books and this one made me feel so much!!! I was happy and mad and sad and then thrilled and then ...
1,Books_5,4.0,CG,"Like almost all of James Patterson's books, the third is a boring, slow read."
2,Clothing_Shoes_and_Jewelry_5,1.0,CG,Im very disappointed with my purchase. The quality is just not what I was expecting.Very pretty.I bought this for my
3,Books_5,5.0,OR,Bought this book as a gift for my wife! Needless to say she loved it!
4,Movies_and_TV_5,5.0,CG,"DVDs were what I expected. The quality of the video is very good, the music is very good, the DVD's are a must have ..."
5,Movies_and_TV_5,1.0,CG,I don't know what agenda this movie is being made. I saw it for the first time last night and I am very happy I did....
6,Clothing_Shoes_and_Jewelry_5,5.0,OR,worked great for my dress to give me sleeves without dying of heat. Originally bought a large but was swimming in i...
7,Clothing_Shoes_and_Jewelry_5,5.0,CG,Love this top. Makes nursing more comfortable and the materials are thick enough to make it comfortable.
8,Clothing_Shoes_and_Jewelry_5,5.0,OR,I have had New Balance cross trainers before and am not disappointed.\nI walk 2 miles daily and they are very comfy....
9,Books_5,3.0,CG,i think nothing can top this book. It is a history book that is well worth the time. I had to read it before going...


,category,rating,label,text_
0,Toys_and_Games_5,1.0,OR,"Incredibly horrible quality and detail. Period. Seriously, it's nothing like the picture."
1,Pet_Supplies_5,2.0,CG,"""MINIMAL"" Attractactant, IF ANY... (Cats are very picky"
2,Clothing_Shoes_and_Jewelry_5,5.0,OR,"Promptly delivers, no qualms about anything else.\n\nGreat boot, I wear it all the time now.\n\nIt works for the off..."
3,Books_5,5.0,CG,What can i say another word about the book? I really enjoyed this book! I am a big fan of the Harry Bosch
4,Home_and_Kitchen_5,2.0,CG,Serious off-gassing!!! ugh! Not only do you have to clean the area around the


In [7]:
! pwd

/Users/svetlana/Documents/CAS_ML/Text/Project/text_analysis/src


In [8]:
# Read the OpenRouter API key from a local file and create the API clien

with open("openrouter_api_key.txt", "r") as f:
    openrouter_api_key = f.read().strip()

print("Key starts with:", openrouter_api_key[:8])
print("Key length:", len(openrouter_api_key))

openai_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

Key starts with: sk-or-v1
Key length: 73


In [9]:
#MODEL = "openai/gpt-4o"

In [10]:
# Select the LLM model used for review classification
MODEL_GPT4O = "openai/gpt-4o"
MODEL_GPT53 = "openai/gpt-5.3-chat"

In [11]:
system_prompt = """Classify sentiment!"""

In [12]:
def ask_llm(user_text, model_name = MODEL_GPT4O):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]

    response = openai_client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0
    )

    return response.choices[0].message.content

In [27]:
answer_4o = ask_llm("Nice weather", model_name=MODEL_GPT4O)
print(answer_4o)

Positive


In [28]:
answer_53 = ask_llm("Nice weather", model_name=MODEL_GPT53)
print(answer_53)

Positive


In [13]:
# Short system prompt: tells the LLM how to classify fake vs. original reviews
system_prompt = """
You are an expert fake-review detector.

Your task is to classify product reviews as either:

CG = computer-generated or fake review
OR = original human-written review

Use the review text, rating, and product category to decide.

Look for signs of CG/fake reviews such as:
- generic language with little specific detail
- repetitive or unnatural wording
- contradictions inside the review
- overly vague praise or criticism
- strange grammar or sentence structure
- review text that feels incomplete or copied
- mismatch between rating and review sentiment
- unrealistic emotional tone
- lack of concrete product experience

Look for signs of OR/real reviews such as:
- specific personal experience
- natural emotional reactions
- concrete details about the product
- realistic imperfections in writing
- clear connection between rating and opinion

Return only valid JSON in this exact format:

{
  "label": "CG or OR"
}

Do not write markdown.
Do not write anything outside the JSON.
"""

In [14]:
# Main system prompt: tells the LLM how to classify fake vs. original reviews
system_prompt_main = """
You are an expert fake-review detector.

Your task is to classify product reviews as either:

CG = computer-generated or fake review
OR = original human-written review

Use the review text, rating, and product category.

Important patterns from this dataset:

1. OR reviews often include specific personal experience.
Example:
Category: Kindle_Store_5
Rating: 5.0
Review: I love paranormal books and this one made me feel so much!!! I was happy and mad and sad and then thrilled and then ...
Label: OR
Reason: Emotional and personal reaction to reading the book.

2. CG reviews can be short, generic, or overly broad.
Example:
Category: Books_5
Rating: 4.0
Review: Like almost all of James Patterson's books, the third is a boring, slow read.
Label: CG
Reason: Generic statement with limited concrete detail.

3. CG reviews may contain unnatural or contradictory text.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 1.0
Review: Im very disappointed with my purchase. The quality is just not what I was expecting.Very pretty.I bought this for my
Label: CG
Reason: The text is incomplete and contradictory.

4. OR reviews may be simple but still natural.
Example:
Category: Books_5
Rating: 5.0
Review: Bought this book as a gift for my wife! Needless to say she loved it!
Label: OR
Reason: Short but believable personal purchasing experience.

5. CG reviews may sound repetitive and promotional.
Example:
Category: Movies_and_TV_5
Rating: 5.0
Review: DVDs were what I expected. The quality of the video is very good, the music is very good, the DVD's are a must have ...
Label: CG
Reason: Repetitive praise and generic wording.

6. CG reviews may have sentiment/rating inconsistency.
Example:
Category: Movies_and_TV_5
Rating: 1.0
Review: I don't know what agenda this movie is being made. I saw it for the first time last night and I am very happy I did....
Label: CG
Reason: Negative rating conflicts with positive wording.

7. OR reviews often mention concrete usage details.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: worked great for my dress to give me sleeves without dying of heat. Originally bought a large but was swimming in i...
Label: OR
Reason: Specific use case and sizing detail.

8. CG reviews may sound smooth but generic.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: Love this top. Makes nursing more comfortable and the materials are thick enough to make it comfortable.
Label: CG
Reason: General praise with limited personal detail.

9. OR reviews often include real-life habits or repeated use.
Example:
Category: Clothing_Shoes_and_Jewelry_5
Rating: 5.0
Review: I have had New Balance cross trainers before and am not disappointed.
I walk 2 miles daily and they are very comfy....
Label: OR
Reason: Specific brand history and daily walking use.

10. CG reviews may contain vague praise that does not fit the rating well.
Example:
Category: Books_5
Rating: 3.0
Review: i think nothing can top this book. It is a history book that is well worth the time. I had to read it before going...
Label: CG
Reason: Strong praise does not match the medium rating.

11. CG reviews may repeat generic positive phrases without giving specific evidence.
Example:
Category: Books_5
Rating: 4.0
Review: I do like this book, and it's an interesting and good read. I would recommend it.I read this book in the middle of ...
Label: CG
Reason: The review uses broad praise like interesting, good read, and recommend it, but gives little concrete detail.

12. OR reviews can be very short when they contain a specific personal habit or use case.
Example:
Category: Electronics_5
Rating: 5.0
Review: Love my p-Touch - i label everything that doesn't move.
Label: OR
Reason: Short but natural, with a specific personal use habit.

Classification rules:
- Predict CG when the review is generic, repetitive, contradictory, incomplete, or inconsistent with the rating.
- Predict OR when the review contains a believable personal experience, concrete use case, specific product detail, or natural emotional reaction.
- Do not assume a review is OR only because it has grammar mistakes.
- Do not assume a review is CG only because it is short.
- Focus on the style and consistency of the review.

Return only valid JSON in this exact format:

{
  "label": "CG or OR"
}

Do not write markdown.
Do not write anything outside the JSON.
"""

In [15]:
def classify_review_json(category, rating, review_text, model_name, system_prompt_text=system_prompt_main, debug=False):
    user_prompt = f"""
Category: {category}
Rating: {rating}
Review: {review_text}
"""

    messages = [
        {"role": "system", "content": system_prompt_text},
        {"role": "user", "content": user_prompt}
    ]

    if debug:
        print(messages)

    response = openai_client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0
    )
    if debug:
        print(response)

    content = response.choices[0].message.content.strip()
    
    return json.loads(content)


In [16]:
def classify_test_reviews(test_df, model_name,system_prompt_text=system_prompt_main, n=50, debug=False):
    test_50 = test_df.head(n).copy()
    rows = [row for _, row in test_50.iterrows()]
    
    # results = test_50.apply(
    #     lambda row: classify_review_json(
    #     category=row["category"],
    #     rating=row["rating"],
    #     review_text=row["text_"],
    #     model_name=model_name,
    #     system_prompt_text=system_prompt_text,
    #     debug=debug,
    #     ),
    #     axis=1
    # )
    results = pqdm(rows, lambda row: classify_review_json(
        category=row["category"],
        rating=row["rating"],
        review_text=row["text_"],
        model_name=model_name,
        system_prompt_text=system_prompt_text,
        debug=debug,
        ), n_jobs=20)

    results = pd.Series(results, index=test_50.index)

    

# def classify_row(row):
#     return classify_review_json(
#         category=row["category"],
#         rating=row["rating"],
#         review_text=row["text_"],
#         model_name=model_name,
#         system_prompt_text=system_prompt_text,
#         debug=debug,
#     )

# rows = [row for _, row in test_50.iterrows()]

# results = pqdm(rows, classify_row, n_jobs=20)

# results = pd.Series(results, index=test_50.index)
    
    test_50["predicted_label"] = results.apply(lambda x: x["label"])
    test_50["explanation"] = results.apply(lambda x: x.get("explanation", ""))

    return test_50[
        ["category", "rating", "label", "predicted_label", "explanation", "text_"]
    ]

In [34]:
##Use GPT-4o:
test_50_gpt4o = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT4O,
    system_prompt_text=system_prompt,
    n=50
)

##test_50_gpt4o

In [17]:
##Use GPT-4o main:
test_50_gpt4o_main = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT4O,
    system_prompt_text=system_prompt_main,
    n=50
)

QUEUEING TASKS | :   0%|          | 0/50 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/50 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/50 [00:00<?, ?it/s]

In [21]:
##Use GPT-4o main:
test_200_gpt4o_main = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT4O,
    system_prompt_text=system_prompt_main,
    n=200
)

QUEUEING TASKS | :   0%|          | 0/200 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/200 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/200 [00:00<?, ?it/s]

In [22]:
##Use GPT-4o main:
test_500_gpt4o_main = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT4O,
    system_prompt_text=system_prompt_main,
    n=500
)

QUEUEING TASKS | :   0%|          | 0/500 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/500 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
#Use GPT-5.3:
test_50_gpt53 = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT53,
    system_prompt_text=system_prompt,
    n=50,
    debug=True
)

In [36]:
##Use GPT-5.3 with main promt:
test_50_gpt53_main = classify_test_reviews(
    test_df,
    model_name=MODEL_GPT53,
    system_prompt_text=system_prompt_main,
    n=50
)

In [19]:
from sklearn.metrics import classification_report

def print_classification_results(result_df):
    # True labels from dataset
    y_true = result_df["label"]

    # Predicted labels from selected LLM model
    y_pred = result_df["predicted_label"]

    # Print precision, recall, and F1 score
    print(classification_report(
        y_true,
        y_pred,
        labels=["CG", "OR"]
    ))

In [56]:
# Print a full classification report
# This shows precision, recall, and F1 score for both CG and OR
print_classification_results(test_50_gpt4o)

              precision    recall  f1-score   support

          CG       0.93      0.65      0.76        20
          OR       0.81      0.97      0.88        30

    accuracy                           0.84        50
   macro avg       0.87      0.81      0.82        50
weighted avg       0.85      0.84      0.83        50



In [20]:
print_classification_results(test_50_gpt4o_main)

              precision    recall  f1-score   support

          CG       0.94      0.75      0.83        20
          OR       0.85      0.97      0.91        30

    accuracy                           0.88        50
   macro avg       0.90      0.86      0.87        50
weighted avg       0.89      0.88      0.88        50



In [23]:
print_classification_results(test_200_gpt4o_main)

              precision    recall  f1-score   support

          CG       0.90      0.79      0.84       101
          OR       0.81      0.91      0.86        99

    accuracy                           0.85       200
   macro avg       0.85      0.85      0.85       200
weighted avg       0.86      0.85      0.85       200



In [24]:
print_classification_results(test_500_gpt4o_main)

              precision    recall  f1-score   support

          CG       0.89      0.82      0.85       255
          OR       0.82      0.89      0.86       245

    accuracy                           0.85       500
   macro avg       0.86      0.85      0.85       500
weighted avg       0.86      0.85      0.85       500



In [55]:
print_classification_results(test_50_gpt53)

              precision    recall  f1-score   support

          CG       0.68      0.85      0.76        20
          OR       0.88      0.73      0.80        30

    accuracy                           0.78        50
   macro avg       0.78      0.79      0.78        50
weighted avg       0.80      0.78      0.78        50



In [54]:
print_classification_results(test_50_gpt53_main)

              precision    recall  f1-score   support

          CG       0.85      0.85      0.85        20
          OR       0.90      0.90      0.90        30

    accuracy                           0.88        50
   macro avg       0.88      0.88      0.88        50
weighted avg       0.88      0.88      0.88        50



In [53]:
# Create a confusion matrix
# It shows where the model was correct and where it made mistakes
cm = confusion_matrix(y_true, y_pred, labels=["CG", "OR"])

# Convert confusion matrix into a readable table
cm_df = pd.DataFrame(
    cm,
    index=["True CG", "True OR"],
    columns=["Predicted CG", "Predicted OR"]
)

# Display the confusion matrix table
cm_df

NameError: name 'confusion_matrix' is not defined